# Armadillo PLY Renderer

In [ ]:
#@title 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 2. Install Dependencies
!wget -nc -q https://download.blender.org/release/Blender4.0/blender-4.0.2-linux-x64.tar.xz
!tar -xf blender-4.0.2-linux-x64.tar.xz
!./blender-4.0.2-linux-x64/4.0/python/bin/python3.10 -m ensurepip --upgrade > /dev/null 2>&1
!./blender-4.0.2-linux-x64/4.0/python/bin/python3.10 -m pip install \
    scipy matplotlib numpy blendertoolbox \
    -q --no-input --disable-pip-version-check

In [ ]:
# @title 3. Download BlenderToolbox
!git clone https://github.com/HTDerekLiu/BlenderToolbox.git
!mv BlenderToolbox/blendertoolbox /content/
!./blender-4.0.2-linux-x64/4.0/python/bin/python3.10 -m pip install blendertoolbox

In [ ]:
#@title 4. Config
PLY_CONFIG = {
    "SAVE_BLEND_FILE": False,
    "FORCE_OVERWRITE": False,
    "DRIVE_BASE_PATH": "/content/drive/MyDrive/PyBlender_Render_Farm",
    "LOCAL_COLAB_BASE": "/content/local_data",
    "PC_TYPE": "armadillo_PLY",
    "OBJ_LOCATION": (0.616392, -0.390241, -0.591646),
    "OBJ_ROTATION": (466.067, -2.68728, -306.609),
    "OBJ_SCALE": (0.006976, 0.006976, 0.006976),
    "IMG_RES_X": 1000, "IMG_RES_Y": 1000,
    "NUM_SAMPLES": 50, "EXPOSURE": 1.5,
    "TILE_SIZE": 256, "SUBDIVISION_LEVEL": 0,
    "CAM_LOCATION": (-1.9494, 1.5553, 0.71451),
    "LOOK_AT": (0, 0, 0), "FOCAL_LENGTH": 45,
    "LIGHT_ANGLE": (-17.5966, -47, -384),
    "LIGHT_STRENGTH": 2, "SHADOW_SOFTNESS": 0.3,
    "AMBIENT_COLOR": (0.1, 0.1, 0.1, 1),
    "SHADOW_THRESHOLD": 0.05,
}
print('Config loaded')

In [ ]:
#@title 5. Write config.py
def _ts(v):
    if isinstance(v, tuple): return '(' + ', '.join(str(x) for x in v) + ')'
    if isinstance(v, list): return '[' + ', '.join(str(x) for x in v) + ']'
    return repr(v)
with open('config.py', 'w') as f:
    f.write('CONFIG = {\n')
    for ck, cv in PLY_CONFIG.items():
        f.write(f'    "{ck}": {_ts(cv)},\n')
    f.write('}\n')
print('config.py written')

In [ ]:
#@title 6. Core Rendering Logic
%%writefile render_ply.py

import os, sys, shutil, gc

if '/content' not in sys.path:
    sys.path.append('/content')
from config import CONFIG

import bpy
import blendertoolbox as bt

def setup_env(CFG, src, local, out):
    print('--- Setting up Environment ---')
    os.makedirs(out, exist_ok=True)
    if os.path.exists(local): shutil.rmtree(local)
    shutil.copytree(src, local)
    n = len([f for f in os.listdir(local) if f.endswith('.ply')])
    print(f'Copy complete! Found {n} .ply files.\n')

def render_single(CFG, meshPath, outputPath):
    sys.stdout.flush()
    print('[STEP] blenderInit...'); sys.stdout.flush()
    bt.blenderInit(CFG['IMG_RES_X'], CFG['IMG_RES_Y'], CFG['NUM_SAMPLES'], CFG['EXPOSURE'])

    print('[STEP] tile_size...'); sys.stdout.flush()
    bpy.context.scene.cycles.tile_size = CFG['TILE_SIZE']

    print('[STEP] readMesh...'); sys.stdout.flush()
    mesh = bt.readMesh(meshPath, CFG['OBJ_LOCATION'], CFG['OBJ_ROTATION'], CFG['OBJ_SCALE'])

    print('[STEP] shade_smooth...'); sys.stdout.flush()
    bpy.ops.object.shade_smooth()

    if CFG['SUBDIVISION_LEVEL'] > 0:
        print(f'[STEP] subdivision level={CFG["SUBDIVISION_LEVEL"]}...'); sys.stdout.flush()
        bt.subdivision(mesh, level=CFG['SUBDIVISION_LEVEL'])
    else:
        print('[STEP] subdivision SKIPPED (level=0)'); sys.stdout.flush()

    print('[STEP] ceramic material...'); sys.stdout.flush()
    meshC = bt.colorObj(bt.derekBlue, 0.5, 1.0, 1.0, 0.0, 0.0)
    subC = bt.colorObj(bt.derekBlue, 0.5, 2.0, 1.0, 0.0, 1.0)
    bt.setMat_ceramic(mesh, meshC, subC)

    print('[STEP] shader node cleanup...'); sys.stdout.flush()
    mat = bpy.context.object.active_material
    nodes = mat.node_tree.nodes
    links = mat.node_tree.links
    mix_shader = next(
        (n for n in nodes if n.type == 'MIX_SHADER'
         and any(o.is_linked and o.links[0].to_node.type == 'OUTPUT_MATERIAL'
                 for o in n.outputs)), None)
    if mix_shader:
        if mix_shader.inputs['Fac'].is_linked:
            for link in mix_shader.inputs['Fac'].links: links.remove(link)
        if mix_shader.inputs[2].is_linked:
            for link in mix_shader.inputs[2].links: links.remove(link)

    print('[STEP] camera & lighting...'); sys.stdout.flush()
    cam = bt.setCamera(CFG['CAM_LOCATION'], CFG['LOOK_AT'], CFG['FOCAL_LENGTH'])
    bt.setLight_sun(CFG['LIGHT_ANGLE'], CFG['LIGHT_STRENGTH'], CFG['SHADOW_SOFTNESS'])
    bt.setLight_ambient(color=CFG['AMBIENT_COLOR'])

    print('[STEP] compositor (simple)...'); sys.stdout.flush()
    bpy.context.scene.use_nodes = True
    tree = bpy.context.scene.node_tree
    tree.nodes.clear()
    rl = tree.nodes.new('CompositorNodeRLayers')
    co = tree.nodes.new('CompositorNodeComposite')
    tree.links.new(rl.outputs['Image'], co.inputs['Image'])

    bt.shadowThreshold(alphaThreshold=CFG['SHADOW_THRESHOLD'], interpolationMode='CARDINAL')

    if CFG['SAVE_BLEND_FILE']:
        bp = outputPath.replace('.png', '.blend')
        bpy.ops.wm.save_mainfile(filepath=bp)

    print('[STEP] renderImage...'); sys.stdout.flush()
    bt.renderImage(outputPath, cam)
    print('[STEP] DONE'); sys.stdout.flush()
    gc.collect()

if __name__ == '__main__':
    C = CONFIG
    print(f"\n{'='*60}")
    print(f"  RENDERING PLY: {C['PC_TYPE']}")
    print(f"{'='*60}\n")
    src = os.path.join(C['DRIVE_BASE_PATH'], 'PointCloud', 'plyFormat', C['PC_TYPE'])
    out = os.path.join(C['DRIVE_BASE_PATH'], 'RenderImages', 'plyFormat', C['PC_TYPE'])
    loc = os.path.join(C['LOCAL_COLAB_BASE'], C['PC_TYPE'])
    setup_env(C, src, loc, out)
    files = sorted([f for f in os.listdir(loc) if f.endswith('.ply')])
    total = len(files)
    print(f'Found {total} .ply files to render.\n')
    for idx, fn in enumerate(files, 1):
        op = os.path.join(out, fn.replace('.ply', '.png'))
        if os.path.exists(op) and not C['FORCE_OVERWRITE']:
            print(f'[{idx}/{total}] Exists. Skipping...'); continue
        print(f'[{idx}/{total}] Rendering [{fn}]...')
        render_single(C, os.path.join(loc, fn), op)
        print(f'[{idx}/{total}] Complete\n')

In [ ]:
#@title 7. Start Rendering
!./blender-4.0.2-linux-x64/blender -b -P render_ply.py

In [ ]:
#@title 8. Extra: Preview Rendered Output
import os
from IPython.display import display, Image
out = '/content/drive/MyDrive/PyBlender_Render_Farm/RenderImages/plyFormat/armadillo_PLY'
if os.path.exists(out):
    pngs = sorted([f for f in os.listdir(out) if f.endswith('.png')])
    print(f'Found {len(pngs)} images')
    for p in pngs[:5]: display(Image(filename=os.path.join(out, p), width=400))
else: print('Not found yet')